<a href="https://colab.research.google.com/github/ahmed-morad15/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-morad15/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [2]:
from huggingface_hub import login
import duckdb

login(token=HF_TOKEN, add_to_git_credential=False)

con = duckdb.connect()

print("DuckDB ready.")
print("Hugging Face authentication ready.")

DuckDB ready.
Hugging Face authentication ready.


In [3]:
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face secret registered in DuckDB.")

Hugging Face secret registered in DuckDB.


In [4]:
import requests

url = "https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main"
response = requests.get(url)

response.raise_for_status()

files = response.json()

[(item["path"], item["type"]) for item in files]

[('fact_content_daily_performance', 'directory'),
 ('.gitattributes', 'file'),
 ('README.md', 'file'),
 ('dim_clients.parquet', 'file'),
 ('dim_content.parquet', 'file'),
 ('fact_content_daily_performance_sample.parquet', 'file'),
 ('fact_content_query_90d.parquet', 'file')]

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis + time window

**Unit of analysis:** One row in the modeling frame represents one content item for one client (`client_hash_id × content_hash_id`) after aggregating the daily performance records.

**Feature window:** February 2026 (2026-02-01 to 2026-02-28). These are the signals that would be knowable at the decision cutoff of February 28.

**Label window:** March 2026 (2026-03-01 to 2026-03-31). This is kept separate from the feature window so that the outcome represents a subsequent period rather than information already known at decision time.

**Decision supported:** The frame supports prioritizing content pages for review based on subsequent search-performance risk.

In [5]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

schema = con.sql(f"""
    DESCRIBE SELECT *
    FROM {REL}
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: feature / label / context / excluded

### Features

The five decision-time features are aggregated from February 2026:

1. `gsc_impressions` — available by the February 28 decision cutoff because it is aggregated only from February GSC daily records.
2. `gsc_clicks` — available by the February 28 decision cutoff because it is aggregated only from February GSC daily records.
3. `gsc_avg_position` — available by the February 28 decision cutoff because it uses only February GSC performance.
4. `ga4_sessions` — available by the February 28 decision cutoff when GA4 data is available for the February records.
5. `ga4_engaged_sessions` — available by the February 28 decision cutoff when GA4 data is available for the February records.

### Label / proxy

`review_priority` is a binary proxy for subsequent search-performance risk. It is derived from March 2026 `gsc_avg_position`, with a page marked for review when its March average position is greater than 10.

The March outcome is used only to construct the label and is never included in the honest feature set.

### Context

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`

These fields are used for grouping, joining, filtering, and aligning time windows. They are not predictive features.

### Excluded

- March performance fields are excluded from the honest feature set because they are future information relative to the February 28 decision cutoff.
- `client_hash_id` and `content_hash_id` are excluded from model inputs because they are pseudonymous identifiers used for grouping and joining.
- Rows with unavailable GSC or GA4 data are not treated as genuine zero activity; the corresponding availability flags must be respected.

### Output

The analysis produces a decision-support feature frame that can be used to prioritize client-content items for review using only information available before the March outcome window.

### Why each feature is available at decision time

| Feature | Available when? |
|---|---|
| `gsc_impressions` | Available by the February 28 decision cutoff because it is aggregated only from February GSC daily records. |
| `gsc_clicks` | Available by the February 28 decision cutoff because it is aggregated only from February GSC daily records. |
| `gsc_avg_position` | Available by the February 28 decision cutoff because it is calculated only from February GSC performance. |
| `ga4_sessions` | Available by the February 28 decision cutoff when GA4 data is available for the February records. |
| `ga4_engaged_sessions` | Available by the February 28 decision cutoff when GA4 data is available for the February records. |

In [6]:
FEB_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
"""

MAR_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [7]:
feature_frame = con.sql(f"""
WITH feb_features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,

        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

    FROM {FEB_REL}

    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

march_outcome AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS march_avg_position

    FROM {MAR_REL}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    f.ga4_sessions,
    f.ga4_engaged_sessions,

    m.march_avg_position,

    CASE
        WHEN m.march_avg_position > 10 THEN 1
        ELSE 0
    END AS review_priority

FROM feb_features f

INNER JOIN march_outcome m
    USING (client_hash_id, content_hash_id)

WHERE f.gsc_avg_position IS NOT NULL
  AND m.march_avg_position IS NOT NULL
""").df()

print("Feature frame shape:", feature_frame.shape)

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (21863, 9)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_avg_position,review_priority
0,client_e547b89c05043229,content_2e296120acb03e93,58.0,0.0,65.896552,2.0,1.0,59.850604,1
1,client_e547b89c05043229,content_dfd3cad4c6495ca6,135.0,0.0,18.155556,3.0,0.0,20.991903,1
2,client_e547b89c05043229,content_07614d4fce7d1cb8,5344.0,26.0,11.688436,32.0,2.0,13.190524,1
3,client_e547b89c05043229,content_a293af8bea1f9d16,1.0,0.0,18.000000,2.0,0.0,18.571429,1
4,client_e547b89c05043229,content_51583fe04ae1be74,3610.0,9.0,8.881163,12.0,0.0,12.996109,1


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

X = feature_frame[feature_cols]
y = feature_frame["review_priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

honest_pred = model.predict(X_test)

honest_accuracy = accuracy_score(y_test, honest_pred)

print(f"Honest accuracy: {honest_accuracy:.4f}")

Honest accuracy: 0.7711


In [9]:
leaky_features = feature_cols + ["march_avg_position"]

X_leaky = feature_frame[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train_l, y_train_l)

leaky_pred = leaky_model.predict(X_test_l)

leaky_accuracy = accuracy_score(y_test_l, leaky_pred)

print(f"Leaky accuracy: {leaky_accuracy:.4f}")

Leaky accuracy: 1.0000


### Leakage lesson

The deliberate leakage experiment adds `march_avg_position`, which is derived from the March outcome window used to define `review_priority`. This information would not be available at the February 28 decision cutoff.

The resulting score is therefore not a valid estimate of model performance. I remove `march_avg_position` from the feature set and retain only the five February decision-time features for the honest analysis.

In [10]:
# Remove the label-derived column from the modeling features.
X_honest = feature_frame[feature_cols].copy()

print("Honest features:")
print(feature_cols)
print("Number of features:", len(feature_cols))

Honest features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']
Number of features: 5


In [11]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS client_content_pairs,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {REL}
WHERE month = '2026-03'
""").df()

q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_dates,client_content_pairs,min_date,max_date
0,9841378,31,331437,2026-03-01,2026-03-31


### Query 2 — March 2026 slice size and date span

This verifies the number of daily records in the March 2026 slice and confirms the observed reporting-date window.

In [12]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {REL}
WHERE month = '2026-03'
""").df()

q2

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — Data availability

This checks how many March 2026 rows have GSC and GA4 data available. I use `IS TRUE` explicitly so NULL values are not treated as available.

In [13]:
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM {REL}
WHERE month = '2026-03'
""").df()

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS client_content_pairs,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {REL}
WHERE month = '2026-03'
""").df()

q1

,total_rows,distinct_dates,client_content_pairs,min_date,max_date
0,9841378,31,331437,2026-03-01,2026-03-31


### Leakage experiment result

The honest model achieved an accuracy of 0.7748 using only the five February decision-time features.

When I deliberately added `march_avg_position`, which is derived from the March outcome window used to define the label, accuracy increased to 0.9998.

This is leakage: the model is given information from the outcome period that would not have been available at the February 28 decision cutoff. The near-perfect score is therefore not a valid estimate of real-world performance.

I removed `march_avg_position` and retained the five February features as the honest feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitation

The main limitation of this slice is uneven source availability. In March 2026, GSC data is available for 3,611,061 of 9,841,378 rows, while GA4 data is available for 413,966 rows, and only 364,347 rows have both sources available.

Therefore, a feature frame that requires both GSC and GA4 cannot represent the full client-content population. The resulting analysis should be treated as decision-support for the observed and available data, not as a complete view of all content.

The panel also has different history depths across clients, so a single calendar window may not represent the same amount of historical information for every client.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.